In [2]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Step 1: Load DINOv2 features from .pickle file
with open('/media/osero/SamsungSSD/pickles/features_left_hand_frames_288.pickle','rb') as f:
    features,labels  = pickle.load(f)

average_features = []
frame_frequency = 5

for feature in features:
    sampled_features = feature[0::frame_frequency]
    average_features.append(np.mean(sampled_features, axis=0))

# Step 2: Convert data to PyTorch tensors
X = torch.tensor(average_features, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)  # Class labels as long for classification

# Step 3: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Create DataLoader for training and testing with pin_memory for faster GPU transfer
train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, pin_memory=True)

# Step 5: Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return self.softmax(x)

# Assuming features are of size (N, D), where D is input_dim
input_dim = X.shape[1]  # Feature dimension
hidden_dim = 128  # Hidden layer dimension (you can adjust this)
output_dim = len(torch.unique(y))  # Number of unique classes

# Initialize the model
model = MLP(input_dim, hidden_dim, output_dim)

# Step 6: Check if CUDA is available and move the model to GPU if possible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Step 7: Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # For multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Step 8: Training loop
num_epochs = 20  # You can adjust the number of epochs

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0

    for batch_features, batch_labels in train_loader:
        # Move data to the GPU
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()  # Zero the gradients

        # Forward pass
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

# Step 9: Evaluate the model on the test set
model.eval()  # Set model to evaluation mode
correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Move data to the GPU
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

accuracy = correct / total * 100
print(f"Test Accuracy: {accuracy:.2f}%")

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call,so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.